# NB05 — 30-Day Wildfire Risk Prediction & Interactive Dashboard

**Goal:** Use the weather forecast from NB03 + the fire model from NB04 to predict **future wildfire risk** for every city × date, then build interactive maps for a jury demo.

### Pipeline
`§1` Load forecast + fire model → `§2` Engineer features → `§3` Predict risk → `§4` Folium maps → `§5` Plotly dashboard → `§6` Weather maps

**Input:** `outputs/weather_forecast_30d.parquet`, `models/wildfire/best_fire_model.joblib`  
**Output:** `outputs/wildfire_risk_30d.parquet`, `reports/maps/*.html`

In [1]:
# ─── Cell 1: Imports ─────────────────────────────────────────────────────
import os, sys, json, warnings
from pathlib import Path

for _p in ["folium", "plotly"]:
    try: __import__(_p)
    except ImportError:
        import subprocess; subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _p])

import numpy as np, pandas as pd
from joblib import load as jl_load
import folium
import plotly.express as px

warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from src.config import (ROOT, PROCESSED, OUTPUTS, MODELS_F, REFERENCE,
                         CITIES, FORECAST_30D, RISK_30D, ENG_DAILY, MASTER_DAILY)

MAPS    = ROOT / "reports" / "maps"
FIGURES = ROOT / "reports" / "figures"
for p in (MAPS, FIGURES): p.mkdir(parents=True, exist_ok=True)
print(f"Root: {ROOT}")

Root: /home/manheim666/Desktop/WildFire-Prediction


In [2]:
# ─── §1: Load forecast, model, historical data ──────────────────────────

# 1a. Weather forecast
assert FORECAST_30D.exists(), f"Run NB03 first — {FORECAST_30D} not found"
fc = pd.read_parquet(FORECAST_30D)
fc["Date"] = pd.to_datetime(fc["Date"])
print(f"Forecast: {fc.shape}  {fc['Date'].min().date()} → {fc['Date'].max().date()}")

# Ensure ALL cities are represented (broadcast if missing)
if "City" not in fc.columns:
    # No City column → broadcast to all cities
    rows = [fc.assign(City=c) for c in CITIES]
    fc = pd.concat(rows, ignore_index=True)
    print(f"  Broadcast to {len(CITIES)} cities → {fc.shape}")
else:
    # City column exists — ensure ALL 16 cities present
    present = set(fc["City"].unique())
    missing = set(CITIES.keys()) - present
    if missing:
        # For missing cities, duplicate the template row (same dates/weather)
        template = fc[fc["City"] == list(present)[0]].copy()
        extras = []
        for city in missing:
            t = template.copy()
            t["City"] = city
            extras.append(t)
        fc = pd.concat([fc] + extras, ignore_index=True)
        print(f"  Added {len(missing)} missing cities: {sorted(missing)}")

fc_cities = sorted(fc["City"].unique())
print(f"  Cities in forecast ({len(fc_cities)}): {fc_cities}")

# 1b. Fire model + manifest
assert (MODELS_F / "best_fire_model.joblib").exists(), "Run NB04 first"
fire_model = jl_load(MODELS_F / "best_fire_model.joblib")

with open(MODELS_F / "model_manifest.json") as f:
    manifest = json.load(f)
THRESHOLD = manifest["optimal_threshold"]
print(f"Model: {manifest['model_name']}  threshold={THRESHOLD:.4f}")

with open(MODELS_F / "feature_columns.json") as f:
    FEATURE_COLS = json.load(f)
print(f"Features: {len(FEATURE_COLS)}")

# 1c. Historical data for lag features
hist_path = ENG_DAILY if ENG_DAILY.exists() else MASTER_DAILY
hist = pd.read_parquet(hist_path)
hist["Date"] = pd.to_datetime(hist["Date"])
hist = hist.sort_values(["City", "Date"])
print(f"History: {hist.shape}")

# 1d. Static geography
static_path = REFERENCE / "static_geography.parquet"
static_df = (pd.read_parquet(static_path) if static_path.exists()
             else pd.DataFrame([{"City": c, "Latitude": lat, "Longitude": lon,
                                  "Elevation": 0, "Slope": 0, "Trees_pct": 0,
                                  "Urban_pct": 0, "Pop_Total": 0}
                                 for c, (lat, lon) in CITIES.items()]))

Forecast: (480, 10)  2026-05-03 → 2026-06-01
  Cities in forecast (16): ['Baku', 'Barda', 'Gabala', 'Ganja', 'Jalilabad', 'Khachmaz', 'Lankaran', 'Mingachevir', 'Nakhchivan', 'Quba', 'Shabran', 'Shaki', 'Shamakhi', 'Shirvan', 'Yevlakh', 'Zaqatala']
Model: XGBoost_Optuna  threshold=0.5000
Features: 184
History: (83487, 269)


In [3]:
# ─── §2: Engineer features for forecast period ──────────────────────────

# Expand forecast to all cities if needed
if "City" not in fc.columns:
    rows = [fc.assign(City=c) for c in CITIES]
    fc = pd.concat(rows, ignore_index=True)
    print(f"Broadcast to {len(CITIES)} cities → {fc.shape}")

# Rename bare weather columns → _mean suffix (match training feature names)
rename_map = {}
for c in fc.columns:
    if c in ("City", "Date"): continue
    if not any(s in c for s in ("_mean","_sum","_min","_max")):
        if c + "_mean" not in fc.columns:
            rename_map[c] = c + "_mean"
if rename_map:
    fc = fc.rename(columns=rename_map)

# Build feature matrix per city (history concat for lags)
frames = []
for city in sorted(fc["City"].unique()):
    cf = fc[fc["City"] == city].sort_values("Date").copy()
    ch = hist[hist["City"] == city].sort_values("Date")
    cutoff = cf["Date"].min() - pd.Timedelta(days=90)
    recent = ch[ch["Date"] >= cutoff]

    common = ["City","Date"] + [c for c in cf.columns
              if c in recent.columns and c not in ("City","Date")]
    combo = pd.concat([recent[common], cf[common]], ignore_index=True)
    combo = combo.sort_values("Date").drop_duplicates(["City","Date"], keep="last")

    # Calendar
    combo["Year"]       = combo["Date"].dt.year
    combo["Month"]      = combo["Date"].dt.month
    combo["DayOfYear"]  = combo["Date"].dt.dayofyear
    combo["DayOfWeek"]  = combo["Date"].dt.dayofweek
    combo["WeekOfYear"] = combo["Date"].dt.isocalendar().week.astype(int)
    for col, period in [("Month",12),("DayOfYear",365),("DayOfWeek",7)]:
        combo[f"{col}_sin"] = np.sin(2*np.pi*combo[col]/period)
        combo[f"{col}_cos"] = np.cos(2*np.pi*combo[col]/period)
    combo["Month_sin"]  = np.sin(2*np.pi*combo["Month"]/12)
    combo["Month_cos"]  = np.cos(2*np.pi*combo["Month"]/12)
    combo["DoY_sin"]    = np.sin(2*np.pi*combo["DayOfYear"]/365)
    combo["DoY_cos"]    = np.cos(2*np.pi*combo["DayOfYear"]/365)
    combo["DoW_sin"]    = np.sin(2*np.pi*combo["DayOfWeek"]/7)
    combo["DoW_cos"]    = np.cos(2*np.pi*combo["DayOfWeek"]/7)
    combo["Season"]     = combo["Month"].map(
        {12:0,1:0,2:0,3:1,4:1,5:1,6:2,7:2,8:2,9:3,10:3,11:3})
    combo["is_summer"]      = combo["Month"].isin([6,7,8]).astype(int)
    combo["is_fire_season"] = combo["Month"].isin([5,6,7,8,9]).astype(int)

    # Lag / rolling from numeric cols
    num_cols = [c for c in combo.columns if combo[c].dtype in
                ("float64","float32","int64") and c not in
                ("City","Date","Year","Month","DayOfYear","DayOfWeek","WeekOfYear")]
    for v in num_cols[:15]:  # cap to top-15 to avoid explosion
        for lag in [1,2,3,7,14,30]:
            combo[f"{v}_lag{lag}"] = combo[v].shift(lag)
        for w in [3,7,14,30]:
            combo[f"{v}_roll{w}_mean"] = combo[v].shift(1).rolling(w,min_periods=1).mean()

    combo = combo.merge(static_df, on="City", how="left")
    frames.append(combo[combo["Date"].isin(cf["Date"].values)])

forecast_df = pd.concat(frames, ignore_index=True).fillna(0)
print(f"Feature matrix: {forecast_df.shape}")

Feature matrix: (480, 187)


In [4]:
# ─── §3: Predict risk ────────────────────────────────────────────────────

# Align columns
for col in FEATURE_COLS:
    if col not in forecast_df.columns:
        forecast_df[col] = 0
X_fc = forecast_df[FEATURE_COLS].fillna(0)
print(f"Prediction matrix: {X_fc.shape}")

fire_proba = fire_model.predict_proba(X_fc)[:, 1]

risk_df = forecast_df[["City", "Date"]].copy()
for col in ["Temperature_C_mean","Humidity_percent_mean","Rain_mm_sum",
            "Wind_Speed_kmh_mean","Pressure_hPa_mean"]:
    if col in forecast_df.columns:
        risk_df[col] = forecast_df[col].values

risk_df["fire_probability"] = fire_proba
risk_df["fire_predicted"]   = (fire_proba >= THRESHOLD).astype(int)
risk_df["risk_level"] = pd.cut(
    fire_proba, bins=[-1, 0.15, 0.35, 0.60, 1.01],
    labels=["Low", "Medium", "High", "Extreme"])

coords = pd.DataFrame([{"City":c,"Latitude":lat,"Longitude":lon}
                        for c,(lat,lon) in CITIES.items()])
risk_df = risk_df.merge(coords, on="City", how="left")

risk_df.to_parquet(RISK_30D, index=False)
risk_df.to_csv(OUTPUTS / "wildfire_risk_30d.csv", index=False)

print(f"\nRisk saved: {risk_df.shape}  → {RISK_30D.name}")
print(f"\nRisk distribution:\n{risk_df['risk_level'].value_counts().to_string()}")
print(f"\nTop 10 highest-risk:")
print(risk_df.nlargest(10, "fire_probability")[
    ["City","Date","fire_probability","risk_level"]].to_string(index=False))
print(f"\nMean risk by city:")
print(risk_df.groupby("City")["fire_probability"].agg(["mean","max"])
      .sort_values("mean", ascending=False).round(4).to_string())

Prediction matrix: (480, 184)

Risk saved: (480, 12)  → wildfire_risk_30d.parquet

Risk distribution:
risk_level
Low        480
Medium       0
High         0
Extreme      0

Top 10 highest-risk:
      City       Date  fire_probability risk_level
      Baku 2026-05-29          0.121146        Low
      Baku 2026-05-30          0.121146        Low
      Baku 2026-05-31          0.121146        Low
      Baku 2026-06-01          0.120886        Low
Nakhchivan 2026-06-01          0.115848        Low
      Baku 2026-05-26          0.105557        Low
      Baku 2026-05-27          0.105557        Low
Nakhchivan 2026-05-20          0.104873        Low
   Shirvan 2026-05-31          0.104350        Low
   Shirvan 2026-06-01          0.104122        Low

Mean risk by city:
               mean     max
City                       
Baku         0.0855  0.1211
Jalilabad    0.0807  0.1005
Lankaran     0.0788  0.0896
Khachmaz     0.0787  0.0888
Nakhchivan   0.0746  0.1158
Shabran      0.0727  0.0911


## §4 — Folium Maps (Per-Date)

Colour-coded circle markers on Azerbaijan. Click for details.

In [5]:
# ─── §4: Folium maps ─────────────────────────────────────────────────────
RISK_COLORS = {"Low":"green","Medium":"orange","High":"red","Extreme":"darkred"}
AZ_CENTER, AZ_ZOOM = [40.4, 47.8], 7

def _folium_map(day_df, date_str):
    m = folium.Map(location=AZ_CENTER, zoom_start=AZ_ZOOM, tiles="CartoDB positron")
    title = (f'<div style="position:fixed;top:10px;left:60px;z-index:1000;'
             f'background:white;padding:10px 20px;border-radius:8px;'
             f'box-shadow:0 2px 6px rgba(0,0,0,.3);font-family:Arial">'
             f'<h3 style="margin:0">Azerbaijan Wildfire Risk — {date_str}</h3></div>')
    m.get_root().html.add_child(folium.Element(title))
    for _, r in day_df.iterrows():
        prob = r.get("fire_probability", 0)
        lvl  = str(r.get("risk_level", "Low"))
        col  = RISK_COLORS.get(lvl, "gray")
        popup = [f"<b>{r['City']}</b><br>Date: {date_str}<br>"
                 f"Fire Risk: {prob*100:.1f}% (<b style='color:{col}'>{lvl}</b>)<br>"]
        for c, lbl, u in [("Temperature_C_mean","Temp","°C"),
                           ("Humidity_percent_mean","Hum","%"),
                           ("Wind_Speed_kmh_mean","Wind"," km/h"),
                           ("Rain_mm_sum","Rain"," mm")]:
            if c in r.index and pd.notna(r[c]):
                popup.append(f"{lbl}: {r[c]:.1f}{u}<br>")
        folium.CircleMarker([r["Latitude"], r["Longitude"]],
            radius=max(8, prob*40), color=col, fill=True,
            fill_color=col, fill_opacity=0.7,
            popup=folium.Popup("".join(popup), max_width=250),
            tooltip=f"{r['City']}: {prob*100:.0f}%").add_to(m)
    legend = ('<div style="position:fixed;bottom:30px;left:30px;z-index:1000;'
              'background:white;padding:10px 15px;border-radius:8px;'
              'box-shadow:0 2px 6px rgba(0,0,0,.3);font-size:12px">'
              '<b>Risk</b><br>'
              '<span style="color:green">●</span> Low<br>'
              '<span style="color:orange">●</span> Medium<br>'
              '<span style="color:red">●</span> High<br>'
              '<span style="color:darkred">●</span> Extreme</div>')
    m.get_root().html.add_child(folium.Element(legend))
    return m

dates = sorted(risk_df["Date"].unique())
key_dates = sorted(set([dates[0]] + list(dates[4::5]) + [dates[-1]]))

for dt in key_dates:
    ds = pd.Timestamp(dt).strftime("%Y-%m-%d")
    dd = risk_df[risk_df["Date"] == dt]
    if dd.empty: continue
    _folium_map(dd, ds).save(str(MAPS / f"fire_risk_{ds}.html"))
    print(f"  ✓ {ds}")

demo = _folium_map(risk_df[risk_df["Date"] == dates[0]],
                   pd.Timestamp(dates[0]).strftime("%Y-%m-%d"))
demo.save(str(MAPS / "fire_risk_demo.html"))
print(f"Demo map: {MAPS / 'fire_risk_demo.html'}")

  ✓ 2026-05-03
  ✓ 2026-05-07


  ✓ 2026-05-12
  ✓ 2026-05-17
  ✓ 2026-05-22
  ✓ 2026-05-27
  ✓ 2026-06-01
Demo map: /home/manheim666/Desktop/WildFire-Prediction/reports/maps/fire_risk_demo.html


## §5 — Plotly Animated Dashboard + Weather Maps

In [6]:
# ─── §5: Plotly dashboard ────────────────────────────────────────────────
plot_df = risk_df.copy()
plot_df["date_str"] = plot_df["Date"].dt.strftime("%Y-%m-%d")
plot_df["fire_pct"] = (plot_df["fire_probability"] * 100).round(1)

fig = px.scatter_geo(
    plot_df, lat="Latitude", lon="Longitude",
    color="fire_pct", size="fire_pct",
    hover_name="City",
    hover_data={"fire_pct":True, "risk_level":True,
                "Latitude":False, "Longitude":False, "date_str":False},
    animation_frame="date_str",
    color_continuous_scale=["green","yellow","orange","red","darkred"],
    range_color=[0, 80], size_max=30,
    title="Azerbaijan Wildfire Risk Forecast — 30-Day Outlook",
)
fig.update_geos(center=dict(lat=40.3,lon=47.8), projection_scale=15,
    showland=True, landcolor="rgb(243,243,243)",
    showocean=True, oceancolor="rgb(204,229,255)",
    showcountries=True)
fig.update_layout(height=700, width=1000,
    coloraxis_colorbar=dict(title="Risk %"))
fig.write_html(str(MAPS / "fire_risk_dashboard.html"), include_plotlyjs="cdn")
print(f"Dashboard: {MAPS / 'fire_risk_dashboard.html'}")
fig.show()

# ── Weather variable maps ────────────────────────────────────────────────
for var, label, cmap in [
    ("Temperature_C_mean",    "Temperature (°C)", "RdYlBu_r"),
    ("Humidity_percent_mean", "Humidity (%)",      "YlGnBu"),
    ("Wind_Speed_kmh_mean",   "Wind (km/h)",       "Purples"),
    ("Rain_mm_sum",           "Rain (mm)",          "Blues"),
]:
    if var not in plot_df.columns: continue
    sub = plot_df.dropna(subset=[var])
    if sub.empty: continue
    fw = px.scatter_geo(
        sub, lat="Latitude", lon="Longitude",
        color=var, size=np.abs(sub[var])+1,
        hover_name="City", animation_frame="date_str",
        color_continuous_scale=cmap, size_max=25,
        title=f"Azerbaijan {label} — 30-Day Forecast",
    )
    fw.update_geos(center=dict(lat=40.3,lon=47.8), projection_scale=15,
        showland=True, landcolor="rgb(243,243,243)", showcountries=True)
    fw.update_layout(height=600, width=900)
    fname = f"weather_{var.split('_')[0].lower()}_dashboard.html"
    fw.write_html(str(MAPS / fname), include_plotlyjs="cdn")
    print(f"  ✓ {label} → {fname}")

print(f"\nAll maps in: {MAPS}")
for f in sorted(MAPS.glob("*.html")):
    print(f"  {f.name} ({f.stat().st_size/1024:.0f} KB)")
print(f"\n→ Next: 06_Climate_Report.ipynb")

Dashboard: /home/manheim666/Desktop/WildFire-Prediction/reports/maps/fire_risk_dashboard.html


  ✓ Temperature (°C) → weather_temperature_dashboard.html
  ✓ Humidity (%) → weather_humidity_dashboard.html
  ✓ Wind (km/h) → weather_wind_dashboard.html
  ✓ Rain (mm) → weather_rain_dashboard.html

All maps in: /home/manheim666/Desktop/WildFire-Prediction/reports/maps
  fire_risk_2026-04-30.html (26 KB)
  fire_risk_2026-05-01.html (24 KB)
  fire_risk_2026-05-02.html (26 KB)
  fire_risk_2026-05-03.html (26 KB)
  fire_risk_2026-05-04.html (26 KB)
  fire_risk_2026-05-05.html (24 KB)
  fire_risk_2026-05-06.html (26 KB)
  fire_risk_2026-05-07.html (26 KB)
  fire_risk_2026-05-09.html (26 KB)
  fire_risk_2026-05-10.html (24 KB)
  fire_risk_2026-05-11.html (26 KB)
  fire_risk_2026-05-12.html (26 KB)
  fire_risk_2026-05-14.html (26 KB)
  fire_risk_2026-05-15.html (24 KB)
  fire_risk_2026-05-16.html (26 KB)
  fire_risk_2026-05-17.html (26 KB)
  fire_risk_2026-05-19.html (26 KB)
  fire_risk_2026-05-20.html (24 KB)
  fire_risk_2026-05-21.html (26 KB)
  fire_risk_2026-05-22.html (26 KB)
  fire_ri